In [ ]:
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

# Step 1: Load the recording
local_path = '/media/server-benshalomlab/T9/Caren/MEASlices_02242025_PVSandCA/MEASlices_02122025_PVSandCA/250212/M07305/Network/000002/data.raw.h5'
recording = se.read_maxwell(local_path, stream_id='well002')

# Step 2: Bandpass filter the recording
freq_min = 300
freq_max = 4500
recording_bp = spre.bandpass_filter(recording, freq_min=freq_min, freq_max=freq_max)

# Step 3: Apply common median reference
recording_cmr = spre.common_reference(recording_bp, reference='global', operator='median')

# Step 4: Slice the recording (e.g., first 300 seconds)
fs = recording.get_sampling_frequency()
recording_chunk = recording_cmr.frame_slice(start_frame=0, end_frame=int(300 * fs))

print(f"Chunk duration: {recording_chunk.get_total_duration()} seconds")

In [ ]:
# Step 5: Select specific channels
# For example, to select channels with IDs 10, 25, and 37 (you can adjust this list)
selected_channel_ids = [942, 966, 963, 959, 939, 898, 883, 871, 847, 843, 683, 863]    # replace with your desired channel IDs

# Convert selected_channel_ids to strings to match the channel_ids in recording_chunk
selected_channel_ids_str = [str(ch) for ch in selected_channel_ids]

# Slice the recording to keep only the selected channels
recording_chunk = recording_chunk.channel_slice(channel_ids=selected_channel_ids_str)

# Confirm selection
print(f"Selected channels: {recording_chunk.get_channel_ids()}")
print(f"New shape (channels x frames): {recording_chunk.get_num_channels()} x {recording_chunk.get_num_frames()}")


In [ ]:
# Step 1: Get channel IDs and locations
channel_ids = recording_chunk.get_channel_ids()
locations = recording_chunk.get_channel_locations()

# Step 2: Map electrodes based on locations
electrodes = []
channel_location_dict = {}
for channel, loc in zip(channel_ids, locations):
    electrode_id = 220 * int(loc[1] / 17.5) + int(loc[0] / 17.5)
    electrodes.append(electrode_id)
    channel_location_dict[channel] = electrode_id

print(f"Mapped {len(electrodes)} electrodes.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Define the channels to analyze
#channels_to_plot = [927, 934, 935, 938, 939, 942, 943, 947, 950, 951, 955, 959, 963, 
                    #966, 967, 971, 974, 975, 979, 983, 987, 991, 995, 
                    #998, 999, 1003, 1007, 1011, 1015]  # Replace with desired channel indices


fs = recording_chunk.get_sampling_frequency()

# Step 2: Extract traces for a specific time range
start_time = 0 # Start time in seconds
end_time = 300    # End time in seconds
traces = recording_chunk.get_traces(
    start_frame=int(start_time * fs),
    end_frame=int(end_time * fs),
    segment_index=0,
    return_scaled=True
)

# Step 3: Detect spikes using a standard deviation threshold
def detect_peaks_stddev(trace, peak_sign, std_multiplier):
    mean = np.mean(trace)
    std = np.std(trace)
    threshold = mean + std_multiplier * std if peak_sign == "pos" else mean - std_multiplier * std
    if peak_sign == "pos":
        peaks = np.where((trace[1:-1] > trace[:-2]) & (trace[1:-1] > trace[2:]) & (trace[1:-1] > threshold))[0] + 1
    elif peak_sign == "neg":
        peaks = np.where((trace[1:-1] < trace[:-2]) & (trace[1:-1] < trace[2:]) & (trace[1:-1] < threshold))[0] + 1
    else:
        raise ValueError("peak_sign must be 'pos' or 'neg'")
    return peaks, threshold

# Step 4: Analyze each channel
peak_sign = "neg"
std_multiplier = 6

# Filter channels_to_plot to include only valid indices
valid_channels_to_plot = [ch for ch in selected_channel_ids if ch < traces.shape[1]]

for channel_idx in selected_channel_ids:
    trace = traces[:, selected_channel_ids.index(channel_idx)]
    peaks, threshold = detect_peaks_stddev(trace, peak_sign, std_multiplier)
    print(f"Channel {selected_channel_ids.index(channel_idx)}: Detected {len(peaks)} spikes")

In [ ]:
import spikeinterface.sorters as ss

In [ ]:
# Step 1: Filter channels_to_plot to include only valid channel indices
valid_channels_to_plot = [ch for ch in selected_channel_ids if ch < traces.shape[1]]
channel_idx = selected_channel_ids.index(channel_idx)
# Step 1: Plot traces with detected spikes
plt.figure(figsize=(12, len(selected_channel_ids) * 2))
for i, channel_idx in enumerate(selected_channel_ids):
    trace = traces[:, selected_channel_ids.index(channel_idx)]
    peaks, _ = detect_peaks_stddev(trace, peak_sign, std_multiplier)
    plt.plot(trace + i * 200, label=f"Channel {channel_idx}")
    plt.plot(peaks, trace[peaks] + i * 200, 'rv', label=f"Spikes Channel {channel_idx}")

plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")
plt.title("Traces with Detected Spikes")
plt.legend()
plt.show()

In [ ]:
# Step 1: Calculate firing rates for each channel
total_duration = recording_chunk.get_total_duration()
firing_rates = {}
for channel_idx in selected_channel_ids:
    # Map channel ID to its corresponding index in the traces array
    channel_index = selected_channel_ids.index(channel_idx)
    trace = traces[:, channel_index]
    peaks, _ = detect_peaks_stddev(trace, peak_sign, std_multiplier)
    firing_rate = len(peaks) / total_duration
    firing_rates[channel_idx] = firing_rate

# Step 2: Display firing rates
for channel_idx, rate in firing_rates.items():
    print(f"Channel {channel_idx}: Firing Rate = {rate:.2f} Hz")

In [ ]:
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw

# Step 1: Select the specific channel (e.g., 972)
channel_id_to_sort = 927
recording_single_channel = recording_chunk.channel_slice(channel_ids=[str(channel_id_to_sort)])

# Step 2: Run spike sorting (e.g., using SpykingCircus)
sorting_output_folder = "spykingcircus_output"
sorting = ss.run_sorter('spykingcircus', recording_single_channel, output_folder=sorting_output_folder)

# Step 3: Plot the traces and detected spikes
# Extract traces for visualization
traces_single_channel = recording_single_channel.get_traces(return_scaled=True)

# Plot the traces
sw.plot_timeseries(recording_single_channel)

# Plot the spike trains
sw.plot_rasters(sorting)

In [ ]:
import spikeinterface.sorters as ss
import spikeinterface.qualitymetrics as qm
default_KS2_params = ss.get_default_sorter_params('kilosort2')
default_KS2_params['detect_threshold'] = 4# Adjust the detection threshold

default_KS2_params['minFR'] = 0 # Adjust the number of workers based on your system


print(default_KS2_params)

output_folder = '/mnt/disk15tb/mmpatil/MEA_Analysis_FEB25/IPNAnalysis/kilosort_output/'

run_sorter = ss.run_sorter('kilosort2',recording=recording_chunk, output_folder=output_folder, docker_image= 'mandarmp/benshalom:v1', verbose=True, remove_existing_folder = True, **default_KS2_params)

 

In [ ]:

sortingKS3 = run_sorter.remove_empty_units()

sortingKS3 = ss.curation.remove_excess_spikes(sortingKS3,recording_chunk)

 

waveform_output_folder = 'waveform_output_05272025'

job_kwargs = dict(n_jobs=32, chunk_duration="1s", progress_bar=True)

waveforms = si.extract_waveforms(recording_chunk,sortingKS3,folder=waveform_output_folder,overwrite=True,**job_kwargs)

 

import spikeinterface.postprocessing as sp

 

sp.compute_spike_amplitudes(waveforms,load_if_exists=True,**job_kwargs)

metrics = qm.compute_quality_metrics(waveforms,load_if_exists=False,**job_kwargs)
